In [1]:
# =============================================================================
# CELL 1 — IMPORTS
# =============================================================================

import yfinance as yf
import pandas as pd
import numpy as np
import datetime
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from statsmodels.tsa.stattools import adfuller, coint
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from scipy.optimize import minimize
from scipy.stats import t as student_t

In [2]:
# =============================================================================
# CELL 2 — DATA
# =============================================================================

def fetch_intraday(ticker, interval="15m", total_days=59):
    end   = datetime.datetime.today()
    start = end - datetime.timedelta(days=total_days)
    df    = yf.download(
        ticker,
        start=start.strftime("%Y-%m-%d"),
        end=end.strftime("%Y-%m-%d"),
        interval=interval,
        auto_adjust=True,
        progress=False
    )
    return df[["Close"]].rename(columns={"Close": ticker})

meta         = fetch_intraday("META")
qqq          = fetch_intraday("QQQ")
data         = pd.concat([meta, qqq], axis=1).dropna()
data.columns = ["META", "QQQ"]

meta_vals = data["META"].values.astype(float)
qqq_vals  = data["QQQ"].values.astype(float)

print(f"Data shape: {data.shape}")
print(f"Window: {data.index[0].date()} → {data.index[-1].date()}")

Data shape: (1040, 2)
Window: 2026-02-27 → 2026-04-24


In [3]:
# =============================================================================
# CELL 3 — KALMAN FILTER (STAGE 1 PARAMETERS)
# =============================================================================

def kalman_filter(meta, qqq, delta=1e-4, Ve=0.001):
    n  = len(meta)
    Vw = delta / (1 - delta) * np.eye(2)

    beta = np.zeros((n, 2))
    P    = np.zeros((n, 2, 2))
    e    = np.zeros(n)
    Q    = np.zeros(n)

    beta[0] = [0, 1]
    P[0]    = np.eye(2)

    for t in range(1, n):
        F         = np.array([1.0, qqq[t]])
        beta_pred = beta[t-1]
        P_pred    = P[t-1] + Vw

        e[t] = meta[t] - F @ beta_pred
        Q[t] = F @ P_pred @ F + Ve

        K       = (P_pred @ F) / Q[t]
        beta[t] = beta_pred + K * e[t]
        P[t]    = P_pred - np.outer(K, F) @ P_pred

    return beta, e, Q

beta, e, Q = kalman_filter(meta_vals, qqq_vals)

print(f"β₁ mean: {beta[:,1].mean():.4f}")
print(f"β₁ std:  {beta[:,1].std():.4f}")
print(f"β₁ range: [{beta[:,1].min():.4f}, {beta[:,1].max():.4f}]")

β₁ mean: 1.0311
β₁ std:  0.0382
β₁ range: [0.9245, 1.1008]


In [4]:
def propagate(particles, delta, nu=5):
    N   = particles.shape[0]
    Vw  = delta / (1 - delta)   # process noise variance (scalar, applied per dim)
    
    # Draw Student-t noise for each particle and each state dimension
    # student_t.rvs(df, scale, size) draws N samples
    noise = student_t.rvs(df=nu, scale=np.sqrt(Vw), size=(N, 2))
    
    return particles + noise

In [5]:
def weight(particles, meta_t, qqq_t, Ve, nu=5):
    N = particles.shape[0]
    
    # Each particle predicts META given QQQ
    # F = [1, qqq_t] — same observation vector as Kalman
    F            = np.array([1.0, qqq_t])
    predictions  = particles @ F          # (N,) — one prediction per particle
    
    # Innovation — how wrong was each particle?
    innovations  = meta_t - predictions   # (N,)
    
    # Weight by Student-t likelihood of each innovation
    # High weight = small innovation = particle predicted well
    log_weights  = student_t.logpdf(
                       innovations,
                       df=nu,
                       scale=np.sqrt(Ve)
                   )                      # (N,) — log weights for numerical stability
    
    # Normalise in log-space to prevent underflow
    log_weights -= np.max(log_weights)    # shift so max = 0
    weights      = np.exp(log_weights)
    weights     /= weights.sum()          # normalise to sum to 1
    
    return weights

In [6]:
def resample(particles, weights):
    N   = len(weights)
    
    # Cumulative sum of weights — like a roulette wheel
    cumsum = np.cumsum(weights)
    
    # Systematic grid — N evenly spaced points with a single random offset
    u0      = np.random.uniform(0, 1/N)
    points  = u0 + np.arange(N) / N     # (N,) — evenly spaced in [0, 1]
    
    # For each point, find which particle it lands on
    indices = np.searchsorted(cumsum, points)   # (N,) — indices into particles
    
    return particles[indices]

In [7]:
def particle_filter(meta, qqq, N=1000, delta=1e-4, Ve=0.001, nu=5):
    """
    Full particle filter for dynamic hedge ratio estimation.
    
    meta  : (T,) array of META prices
    qqq   : (T,) array of QQQ prices
    N     : number of particles
    delta : process noise scaling
    Ve    : observation noise variance
    nu    : Student-t degrees of freedom
    
    Returns:
        beta_pf : (T, 2) array of hedge ratio estimates [β₀, β₁]
        spreads : (T,) array of innovations (spread)
    """
    T = len(meta)
    
    # --- Initialise ---
    # Particles drawn from prior — centred on [0, 1] with small spread
    particles = np.zeros((N, 2))
    particles[:, 0] = np.random.normal(0, 1, N)      # β₀ prior
    particles[:, 1] = np.random.normal(1, 0.1, N)    # β₁ prior — centred on 1
    
    # Storage
    beta_pf = np.zeros((T, 2))
    spreads = np.zeros(T)
    
    # --- Bar 0 — initialise estimate from prior ---
    weights       = np.ones(N) / N        # equal weights at start
    beta_pf[0]    = np.average(particles, weights=weights, axis=0)
    spreads[0]    = 0.0
    
    # --- Main loop ---
    for t in range(1, T):
        # Step 1 — Propagate
        particles = propagate(particles, delta, nu)
        
        # Step 2 — Weight
        weights = weight(particles, meta[t], qqq[t], Ve, nu)
        
        # Step 3 — Resample
        particles = resample(particles, weights)
        
        # Estimate hedge ratio as mean of resampled particles
        beta_pf[t] = particles.mean(axis=0)
        
        # Spread = META - predicted META using current hedge ratio
        F          = np.array([1.0, qqq[t]])
        spreads[t] = meta[t] - F @ beta_pf[t]
    
    return beta_pf, spreads

In [9]:
# =============================================================================
# CELL 5 — MLE CALIBRATION OF KALMAN NOISE PARAMETERS
# =============================================================================

def run_kalman_loglik(params, meta, qqq):
    delta = np.exp(params[0])
    Ve    = np.exp(params[1])
    n     = len(meta)
    Vw    = delta / (1 - delta) * np.eye(2)

    beta    = np.zeros((n, 2))
    P       = np.zeros((n, 2, 2))
    log_lik = 0.0

    beta[0] = [0, 1]
    P[0]    = np.eye(2)

    for t in range(1, n):
        F         = np.array([1.0, qqq[t]])
        beta_pred = beta[t-1]
        P_pred    = P[t-1] + Vw

        e_t = meta[t] - F @ beta_pred
        Q_t = F @ P_pred @ F + Ve

        log_lik += -0.5 * (np.log(Q_t) + e_t**2 / Q_t)

        K       = (P_pred @ F) / Q_t
        beta[t] = beta_pred + K * e_t
        P[t]    = P_pred - np.outer(K, F) @ P_pred

    return -log_lik

# Optimise from Stage 1 starting point
x0     = [np.log(1e-4), np.log(0.001)]
result = minimize(
    run_kalman_loglik,
    x0,
    args=(meta_vals, qqq_vals),
    method="Nelder-Mead",
    options={"maxiter": 10000, "xatol": 1e-6, "fatol": 1e-6}
)

delta_mle = np.exp(result.x[0])
Ve_mle    = np.exp(result.x[1])

# Rerun filter with MLE parameters
beta_mle, e_mle, Q_mle = kalman_filter(meta_vals, qqq_vals,
                                        delta=delta_mle, Ve=Ve_mle)

adf_mle = adfuller(e_mle, autolag="AIC")


In [10]:
# =============================================================================
# CELL 4 — PARTICLE FILTER
# =============================================================================

from scipy.stats import t as student_t

# --- propagate, weight, resample, particle_filter functions here ---

np.random.seed(42)
beta_pf, spreads_pf = particle_filter(
    meta_vals, qqq_vals,
    N=1000, delta=delta_mle, Ve=Ve_mle, nu=5
)

print("=== Particle Filter — Hedge Ratio ===")
print(f"β₁ mean:  {beta_pf[:,1].mean():.4f}")
print(f"β₁ std:   {beta_pf[:,1].std():.4f}")
print(f"β₁ range: [{beta_pf[:,1].min():.4f}, {beta_pf[:,1].max():.4f}]")

print("\n=== Kalman Filter — Hedge Ratio (for comparison) ===")
print(f"β₁ mean:  {beta_mle[:,1].mean():.4f}")
print(f"β₁ std:   {beta_mle[:,1].std():.4f}")
print(f"β₁ range: [{beta_mle[:,1].min():.4f}, {beta_mle[:,1].max():.4f}]")

# ADF on particle filter spread
adf_pf = adfuller(spreads_pf[50:], autolag="AIC")
print(f"\n=== Particle Filter Spread Stationarity ===")
print(f"ADF p-value: {adf_pf[1]:.4f} — stationary: {adf_pf[1] < 0.05}")

=== Particle Filter — Hedge Ratio ===
β₁ mean:  1.0297
β₁ std:   0.0382
β₁ range: [0.9230, 1.0995]

=== Kalman Filter — Hedge Ratio (for comparison) ===
β₁ mean:  1.0313
β₁ std:   0.0382
β₁ range: [0.9247, 1.1008]

=== Particle Filter Spread Stationarity ===
ADF p-value: 0.0000 — stationary: True


In [11]:
# =============================================================================
# CELL 5 — HEDGE RATIO COMPARISON: KALMAN VS PARTICLE FILTER
# =============================================================================

# Align both hedge ratios with data index
beta_kalman_series = pd.Series(beta_mle[:, 1],  index=data.index)
beta_pf_series     = pd.Series(beta_pf[:, 1],   index=data.index)

# Difference between the two
beta_diff = beta_pf_series - beta_kalman_series

# -----------------------------------------------------------------------------
# 5A — OVERALL COMPARISON
# -----------------------------------------------------------------------------
print("=== 5A — Overall Hedge Ratio Comparison ===")
print(f"{'Metric':<30} {'Kalman':<15} {'Particle':<15} {'Difference'}")
print("-" * 70)
print(f"{'β₁ mean':<30} {beta_kalman_series.mean():<15.4f} "
      f"{beta_pf_series.mean():<15.4f} "
      f"{beta_diff.mean():.4f}")
print(f"{'β₁ std':<30} {beta_kalman_series.std():<15.4f} "
      f"{beta_pf_series.std():<15.4f} "
      f"{beta_diff.std():.4f}")
print(f"{'β₁ min':<30} {beta_kalman_series.min():<15.4f} "
      f"{beta_pf_series.min():<15.4f} "
      f"{beta_diff.min():.4f}")
print(f"{'β₁ max':<30} {beta_kalman_series.max():<15.4f} "
      f"{beta_pf_series.max():<15.4f} "
      f"{beta_diff.max():.4f}")
print(f"{'Mean absolute difference':<30} {beta_diff.abs().mean():.6f}")

# -----------------------------------------------------------------------------
# 5B — DIVERGENCE DURING TARIFF SHOCK
# The key test — do filters diverge during extreme events?
# -----------------------------------------------------------------------------
normal  = (data.index < "2026-04-07") | (data.index > "2026-04-10")
tariff  = (data.index >= "2026-04-07") & (data.index <= "2026-04-10")

print("\n=== 5B — Divergence by Regime ===")
print(f"{'Metric':<35} {'Normal':<15} {'Tariff Shock'}")
print("-" * 60)

for label, mask in [("Normal", normal), ("Tariff shock", tariff)]:
    diff = beta_diff[mask]
    print(f"Mean |Kalman - Particle| ({label}):  "
          f"{diff.abs().mean():.6f}")
    print(f"Max  |Kalman - Particle| ({label}):  "
          f"{diff.abs().max():.6f}")

# -----------------------------------------------------------------------------
# 5C — BAR BY BAR DURING TARIFF SHOCK
# Shows exactly when and how much the filters diverge
# -----------------------------------------------------------------------------
tariff_window = data.index[tariff]

print(f"\n=== 5C — Bar-by-Bar During Tariff Shock ===")
print(f"{'Datetime':<35} {'Kalman β₁':<15} {'PF β₁':<15} {'Difference'}")
print("-" * 70)

for idx in tariff_window[:20]:  # first 20 bars of shock
    k = beta_kalman_series[idx]
    p = beta_pf_series[idx]
    d = p - k
    print(f"{str(idx):<35} {k:<15.4f} {p:<15.4f} {d:.4f}")

# -----------------------------------------------------------------------------
# 5D — HEDGE RATIO ADAPTATION SPEED
# How many bars does each filter take to reach minimum β₁?
# Faster = better regime detection
# -----------------------------------------------------------------------------
shock_start = pd.Timestamp("2026-04-07", tz="UTC")

kalman_shock = beta_kalman_series[tariff]
pf_shock     = beta_pf_series[tariff]

kalman_min_bar = kalman_shock.idxmin()
pf_min_bar     = pf_shock.idxmin()

print(f"\n=== 5D — Adaptation Speed ===")
print(f"Kalman minimum β₁:          {kalman_shock.min():.4f} "
      f"at {kalman_min_bar}")
print(f"Particle Filter minimum β₁: {pf_shock.min():.4f} "
      f"at {pf_min_bar}")
print(f"Kalman β₁ at shock start:   "
      f"{beta_kalman_series[beta_kalman_series.index >= shock_start].iloc[0]:.4f}")
print(f"PF β₁ at shock start:       "
      f"{beta_pf_series[beta_pf_series.index >= shock_start].iloc[0]:.4f}")

=== 5A — Overall Hedge Ratio Comparison ===
Metric                         Kalman          Particle        Difference
----------------------------------------------------------------------
β₁ mean                        1.0313          1.0297          -0.0016
β₁ std                         0.0382          0.0383          0.0008
β₁ min                         0.9247          0.9230          -0.0223
β₁ max                         1.1008          1.0995          0.0071
Mean absolute difference       0.001642

=== 5B — Divergence by Regime ===
Metric                              Normal          Tariff Shock
------------------------------------------------------------
Mean |Kalman - Particle| (Normal):  0.001595
Max  |Kalman - Particle| (Normal):  0.007084
Mean |Kalman - Particle| (Tariff shock):  0.002224
Max  |Kalman - Particle| (Tariff shock):  0.022265

=== 5C — Bar-by-Bar During Tariff Shock ===
Datetime                            Kalman β₁       PF β₁           Difference
------------

In [12]:
# =============================================================================
# 5E — DELTA SENSITIVITY: WHERE DOES DIVERGENCE BECOME MEANINGFUL?
# =============================================================================
# Test both filters at larger delta values
# Hypothesis: at higher delta, particle filter diverges more during shock
# =============================================================================

deltas_test = [1e-5, 1e-4, 5e-4, 1e-3]

print("=== 5E — Filter Divergence vs Delta ===")
print(f"{'Delta':<12} {'Normal MAD':<18} {'Shock MAD':<18} "
      f"{'Shock/Normal':<15} {'KF min β₁':<12} {'PF min β₁'}")
print("-" * 85)

for d in deltas_test:
    # Kalman
    b_kf, e_kf, _ = kalman_filter(meta_vals, qqq_vals, delta=d, Ve=Ve_mle)
    kf_series      = pd.Series(b_kf[:, 1], index=data.index)

    # Particle filter
    np.random.seed(42)
    b_pf, _    = particle_filter(meta_vals, qqq_vals,
                                  N=1000, delta=d, Ve=Ve_mle, nu=5)
    pf_series  = pd.Series(b_pf[:, 1], index=data.index)

    diff       = (pf_series - kf_series).abs()
    normal_mad = diff[normal].mean()
    shock_mad  = diff[tariff].mean()
    ratio      = shock_mad / normal_mad if normal_mad > 0 else np.nan
    kf_min     = kf_series[tariff].min()
    pf_min     = pf_series[tariff].min()

    print(f"{d:<12} {normal_mad:<18.6f} {shock_mad:<18.6f} "
          f"{ratio:<15.2f} {kf_min:<12.4f} {pf_min:.4f}")

=== 5E — Filter Divergence vs Delta ===
Delta        Normal MAD         Shock MAD          Shock/Normal    KF min β₁    PF min β₁
-------------------------------------------------------------------------------------
1e-05        0.001732           0.002441           1.41            0.9726       0.9706
0.0001       0.000297           0.000518           1.74            0.9723       0.9729
0.0005       0.001138           0.001369           1.20            0.9723       0.9738
0.001        0.000937           0.000288           0.31            0.9723       0.9723
